# Foundation-nnU-Net — Training Notebook

**For local or Colab use:**
1. If on Colab: Runtime → Change runtime type → **T4 GPU**
2. Run cells top to bottom

**On disconnect/crash:** re-run all cells — training resumes from `checkpoints/last_*.pth`

## Cell 1 — Verify GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected — training will be very slow on CPU')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 2 — Set project directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Foundation-nnU-Net/foundation-nnunet'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\beko5\\Desktop\\Foundation-nnU-Net\\foundation-nnunet'

## Cell 3 — Install dependencies

In [ ]:
!pip install -q timm albumentations pydicom scikit-learn tqdm pyyaml scipy opencv-python-headless
print('Dependencies installed.')

## Cell 4 — Verify data

In [ ]:
import json
from pathlib import Path

processed = Path('data/processed/pneumothorax')
splits_path = processed / 'splits.json'

assert splits_path.exists(), 'splits.json not found — run preprocess.py first'
with open(splits_path) as f:
    splits = json.load(f)

n_images = len(list((processed / 'images').glob('*.png')))
n_masks  = len(list((processed / 'masks').glob('*.png')))

print(f'Images: {n_images} | Masks: {n_masks}')
print(f'Train: {len(splits["train"])} | Val: {len(splits["val"])} | Test: {len(splits["test"])}')
assert n_images == n_masks, 'Image/mask count mismatch!'
print('Data OK.')

## Cell 5 — Configure: Baseline training

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['model']['type']            = 'baseline'
cfg['data']['input_size']       = 256
cfg['training']['batch_size']   = 16
cfg['training']['epochs']       = 150
cfg['device']                   = 'auto'
cfg['data']['num_workers']      = 2

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config set to BASELINE — 256x256, bs=16, 150 epochs')
print(yaml.dump(cfg, default_flow_style=False))

## Cell 6 — Train Baseline

In [ ]:
!python -m src.training.trainer --config configs/config.yaml

## Cell 7 — Configure: Hybrid training

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['model']['type']          = 'hybrid'
cfg['data']['input_size']     = 256
cfg['training']['batch_size'] = 8    # hybrid uses more VRAM (Foundation X backbone)
cfg['training']['epochs']     = 150
cfg['device']                 = 'auto'
cfg['data']['num_workers']    = 2

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config set to HYBRID — 256x256, bs=8, 150 epochs')
print(yaml.dump(cfg, default_flow_style=False))

## Cell 8 — Train Hybrid

In [ ]:
!python -m src.training.trainer --config configs/config.yaml

## Cell 9 — Evaluate both models on test set

In [ ]:
print('=== Evaluating Baseline ===')
!python -m src.evaluation.evaluate \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_baseline.pth \
    --model_type baseline

print('\n=== Evaluating Hybrid ===')
!python -m src.evaluation.evaluate \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_hybrid.pth \
    --model_type hybrid

## Cell 10 — Generate visualizations

In [ ]:
# Generate all plots
!python -m src.evaluation.visualize \
    --results_dir results/ \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_baseline.pth \
    --model_type baseline

# Also generate hybrid predictions
!python -m src.evaluation.visualize \
    --results_dir results/ \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_hybrid.pth \
    --model_type hybrid

# Show results inline
from IPython.display import Image, display
from pathlib import Path

for img in sorted(Path('results').glob('*.png')):
    print(f'\n--- {img.name} ---')
    display(Image(str(img)))

## Cell 11 — Final summary

In [ ]:
import pandas as pd
from pathlib import Path

print('=' * 60)
print('TRAINING COMPLETE — FINAL RESULTS')
print('=' * 60)

for model_type in ['baseline', 'hybrid']:
    metrics_file = Path(f'results/test_metrics_{model_type}.csv')
    if metrics_file.exists():
        df = pd.read_csv(metrics_file)
        print(f'\n--- {model_type.upper()} ---')
        print(f'  Test Dice:  {df["dice"].mean():.4f} ± {df["dice"].std():.4f}')
        print(f'  Test IoU:   {df["iou"].mean():.4f} ± {df["iou"].std():.4f}')
        if 'positive' in df.columns:
            pos = df[df['positive'] == True]
            print(f'  Positive Dice: {pos["dice"].mean():.4f} ± {pos["dice"].std():.4f} (n={len(pos)})')
    else:
        print(f'\n--- {model_type.upper()}: not evaluated yet ---')

print('\n' + '=' * 60)
print('All results saved in results/ folder.')
print('Checkpoints saved in checkpoints/ folder.')